In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
    explained_variance_score
)
import optuna
from sklearn.model_selection import cross_val_score

from scipy.stats import pearsonr
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline


In [17]:
data_df = pd.read_csv("D:\APSirCNTandLSTM\CNT with LSTM\Datasets\CNT_Reinforced Concrete_Final Dataset.csv") # Read the data

data_df.dropna(inplace = True) # Drop missing/null values

print("Original Data Shape: ", data_df.shape)

def is_numeric_row(row):
    try:
        pd.to_numeric(row, errors='raise')
        return True
    except:
        return False
    

numeric_rows = data_df.apply(lambda row: is_numeric_row(row), axis=1)
data_df = data_df[numeric_rows]

print("Data Shape (rows, cols): ", data_df.shape) # Print data shape
data_df # Print/visualize the first 5 rows of the data

Original Data Shape:  (295, 7)
Data Shape (rows, cols):  (295, 7)


,Cement (kg/m³),w/c,Coarse aggregate (kg/m³),Fine aggregate (kg/m³),Curing (days),CNT (%),Compressive strength(MPa)
0,360.0,0.50,950.0,745.0,28,0.00,39.500
1,320.0,0.56,898.0,782.0,28,0.00,40.000
2,328.5,0.68,533.7,625.1,28,0.00,40.000
3,360.0,0.50,898.0,797.0,28,0.00,40.000
4,320.0,0.56,845.0,834.0,28,0.00,40.300
...,...,...,...,...,...,...,...
290,400.0,0.45,987.0,780.0,28,0.05,64.000
291,450.0,0.35,1006.0,768.0,28,0.05,84.900
292,525.0,0.45,0.0,1580.0,28,0.00,29.875
293,525.0,0.45,0.0,1580.0,28,0.06,38.750


In [18]:
#Converting non-numeric columns to numeric, if any. This will convert non-numeric values to NaN, which will be dropped later.

for column in data_df.columns:
    data_df[column] = pd.to_numeric(data_df[column], errors='coerce')


target_column = "Compressive strength(MPa)"

# Check data types
print("Data types:")
print(data_df.dtypes)
print("\n" + "="*50 + "\n")

# Check if target column has issues
print(f"Target column '{target_column}' unique values (first 10):")
print(data_df[target_column].unique()[:10])
print(f"\nTarget column type: {data_df[target_column].dtype}")

# Check for any remaining object columns
object_cols = data_df.select_dtypes(include=['object']).columns.tolist()
print(f"\nObject (string) columns: {object_cols}")

Data types:
Cement (kg/m³)               float64
w/c                          float64
Coarse aggregate (kg/m³)     float64
Fine aggregate (kg/m³)       float64
Curing (days)                  int64
CNT (%)                      float64
Compressive strength(MPa)    float64
dtype: object


Target column 'Compressive strength(MPa)' unique values (first 10):
[39.5 40.  40.3 40.8 41.  41.1 41.3 41.5 41.8 42. ]

Target column type: float64

Object (string) columns: []


In [19]:
# Prepare Input and Output

X = np.array(data_df.iloc[:, :-1])
Y = np.array(data_df.iloc[:, -1])

print("Input Shape:", X.shape)
print("Output Shape:", Y.shape)


Input Shape: (295, 6)
Output Shape: (295,)


In [20]:
X, Y

(array([[3.600e+02, 5.000e-01, 9.500e+02, 7.450e+02, 2.800e+01, 0.000e+00],
        [3.200e+02, 5.600e-01, 8.980e+02, 7.820e+02, 2.800e+01, 0.000e+00],
        [3.285e+02, 6.800e-01, 5.337e+02, 6.251e+02, 2.800e+01, 0.000e+00],
        ...,
        [5.250e+02, 4.500e-01, 0.000e+00, 1.580e+03, 2.800e+01, 0.000e+00],
        [5.250e+02, 4.500e-01, 0.000e+00, 1.580e+03, 2.800e+01, 6.000e-02],
        [5.250e+02, 4.500e-01, 0.000e+00, 1.580e+03, 2.800e+01, 1.200e-01]]),
 array([39.5  , 40.   , 40.   , 40.   , 40.3  , 40.8  , 40.8  , 41.   ,
        41.1  , 41.3  , 41.3  , 41.5  , 41.5  , 41.5  , 41.5  , 41.8  ,
        42.   , 42.1  , 42.4  , 42.5  , 43.   , 43.2  , 43.3  , 43.5  ,
        43.6  , 43.8  , 44.   , 44.5  , 45.   , 45.   , 45.1  , 45.7  ,
        46.   , 46.   , 47.1  , 47.7  , 50.   , 36.8  , 43.1  , 38.6  ,
        47.2  , 45.1  , 49.6  , 47.4  , 54.   , 50.1  , 37.8  , 44.1  ,
        40.9  , 47.5  , 45.3  , 42.5  , 52.   , 48.7  , 46.6  , 54.5  ,
        53.1  , 49.2  , 4

In [21]:

# 80 : 20 Split

X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.20, random_state=42
)

print("Training X:", X_train.shape)
print("Testing X:", X_test.shape)

print("Training y:", y_train.shape)
print("Testing y:", y_test.shape)


print("Training set is: ", X_train.shape[0], " rows which is ", round(X_train.shape[0]/data_df.shape[0],4)*100, "%") # Print training shape
print("Testing set is: ",X_test.shape[0], " rows which is ", round(X_test.shape[0]/data_df.shape[0],4)*100, "%") # Print testing shape


Training X: (236, 6)
Testing X: (59, 6)
Training y: (236,)
Testing y: (59,)
Training set is:  236  rows which is  80.0 %
Testing set is:  59  rows which is  20.0 %


In [22]:
scaler_X_XGB = StandardScaler()

X_train_scaled = scaler_X_XGB.fit_transform(X_train)

X_test_scaled = scaler_X_XGB.transform(X_test)

print("XGB Training Shape X:", X_train_scaled.shape)
print("XGB testing Shape X:", X_test_scaled.shape)
print("**********************************")
print("XGB Training Shape y:", y_train.shape)
print("XGB testing Shape y:", y_test.shape)


XGB Training Shape X: (236, 6)
XGB testing Shape X: (59, 6)
**********************************
XGB Training Shape y: (236,)
XGB testing Shape y: (59,)


In [23]:
# Model

xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,

    objective='reg:squarederror',
    random_state=42
)

# Training

xgb_model.fit(
    X_train_scaled,
    y_train,
    verbose=False
)


# Prediction

y_pred_train = xgb_model.predict(X_train_scaled)
y_pred_test  = xgb_model.predict(X_test_scaled)

print(y_pred_test)
print(y_pred_train)


[63.07724  57.175415 53.434917 51.308594 38.489105 41.157818 51.23008
 52.483402 48.606094 53.23657  53.378708 33.300976 44.569702 51.57793
 59.912045 37.300545 51.438282 31.688866 19.612022 52.765755 61.37308
 59.30876  54.965797 58.837914 42.24055  43.326336 39.53917  15.947409
 44.022625 39.318844 23.295103 44.265697 18.902142 18.902142 38.405502
 40.379097 60.556255 55.33811  50.22905  33.310635 24.120142 52.743484
 48.606716 51.63046  45.741486 43.272545 52.22973  39.146442 39.989025
 45.67664  49.335674 59.135963 43.002827 42.30973  13.022731 44.343555
 51.438282 44.538708 55.33811 ]
[46.079777 39.404057 42.499928 44.65982  19.612022 59.610867 48.393356
 53.65363  47.550064 40.379097 54.427734 61.37308  49.667175 42.269638
 50.83354  39.84088  56.71758  34.812782 41.5873   41.825966 55.960716
 45.497967 58.37207  37.300545 41.857857 30.448833 53.378708 46.702557
 47.730618 51.04888  51.986084 47.550064 55.588882 43.47956  52.137505
 57.97281  41.932434 53.965393 50.055885 49.1934

In [24]:
def evaluate_regression(y_true, y_pred):

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    r2 = r2_score(y_true, y_pred)

    mse = mean_squared_error(y_true, y_pred)

    rmse = np.sqrt(mse)

    mae = mean_absolute_error(y_true, y_pred)

    # avoid divide-by-zero
    mape = np.mean(
        np.abs((y_true - y_pred) / (y_true + 1e-8))
    ) * 100

    evs = explained_variance_score(y_true, y_pred)

    pearson_r, _ = pearsonr(y_true, y_pred)

    relative_rmse = (rmse / np.mean(y_true)) * 100

    metrics = {
        "R2": r2,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE (%)": mape,
        "Explained Variance Score": evs,
        "Pearson Correlation": pearson_r,
        "Relative RMSE (%)": relative_rmse
    }

    return metrics

print("TRAIN METRICS")
print(evaluate_regression(y_train, y_pred_train))


print("\nTEST METRICS")
print(evaluate_regression(y_test, y_pred_test))

TRAIN METRICS
{'R2': 0.9923221465841879, 'MSE': 0.8277152206878786, 'RMSE': 0.9097885582309104, 'MAE': 0.6344924430523889, 'MAPE (%)': 1.5338048628853258, 'Explained Variance Score': 0.9923225099043823, 'Pearson Correlation': 0.9961650656695328, 'Relative RMSE (%)': 1.99320466837721}

TEST METRICS
{'R2': 0.9578139969876052, 'MSE': 6.123274458105095, 'RMSE': 2.474525097489434, 'MAE': 1.748742175991252, 'MAPE (%)': 4.849644772584811, 'Explained Variance Score': 0.9591314590416502, 'Pearson Correlation': 0.9794347688373466, 'Relative RMSE (%)': 5.597979346551866}


In [25]:
import pandas as pd

train_metrics = evaluate_regression(y_train, y_pred_train)
test_metrics  = evaluate_regression(y_test, y_pred_test)

df_results = pd.DataFrame({
    "Train": train_metrics,
    "Test": test_metrics
})

print(df_results.round(4))

                           Train    Test
R2                        0.9923  0.9578
MSE                       0.8277  6.1233
RMSE                      0.9098  2.4745
MAE                       0.6345  1.7487
MAPE (%)                  1.5338  4.8496
Explained Variance Score  0.9923  0.9591
Pearson Correlation       0.9962  0.9794
Relative RMSE (%)         1.9932  5.5980


In [26]:

# OPTUNA + XGBOOST + 5-FOLD CV

# K-Fold

kfold = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Objective function
def objective(trial):

    params = {

        "n_estimators": trial.suggest_int(
            "n_estimators",
            100,
            900
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.1,
            log=True
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            3,
            8
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight",
            1,
            6
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.6,
            1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.6,
            1.0
        ),

        "gamma": trial.suggest_float(
            "gamma",
            0,
            0.3
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            0,
            1
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            0.1,
            10,
            log=True
        ),

        "objective": "reg:squarederror",

        "random_state": 42
    }


   
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", XGBRegressor(**params))
    ])


    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,

        cv=kfold,

        scoring="r2",

        n_jobs=-1
    )


    return scores.mean()

In [27]:
study = optuna.create_study(
    direction="maximize",
    study_name="xgb_study",
)

study.optimize(
    objective,
    n_trials=50
)

[I 2026-06-03 09:05:58,966] A new study created in memory with name: xgb_study
[I 2026-06-03 09:05:59,144] Trial 0 finished with value: 0.8490334201919207 and parameters: {'n_estimators': 596, 'learning_rate': 0.020899305388209343, 'max_depth': 8, 'min_child_weight': 2, 'subsample': 0.9845513350339116, 'colsample_bytree': 0.9933670818162842, 'gamma': 0.05355931656245964, 'reg_alpha': 0.12555804022654438, 'reg_lambda': 0.12176115387443895}. Best is trial 0 with value: 0.8490334201919207.
[I 2026-06-03 09:05:59,202] Trial 1 finished with value: 0.7963504335620392 and parameters: {'n_estimators': 289, 'learning_rate': 0.023304116675169642, 'max_depth': 3, 'min_child_weight': 4, 'subsample': 0.9707819918494832, 'colsample_bytree': 0.9441630828012916, 'gamma': 0.006929290470467608, 'reg_alpha': 0.6569538533736692, 'reg_lambda': 1.0450830603213448}. Best is trial 0 with value: 0.8490334201919207.
[I 2026-06-03 09:05:59,250] Trial 2 finished with value: 0.8236399683306084 and parameters: {'n_

In [28]:
print("Best Parameters:")
print(study.best_params)

print("\nBest CV R2:")
print(study.best_value)

Best Parameters:
{'n_estimators': 886, 'learning_rate': 0.09639327381314948, 'max_depth': 4, 'min_child_weight': 2, 'subsample': 0.6296036837878618, 'colsample_bytree': 0.6988253984127397, 'gamma': 0.1774745719782075, 'reg_alpha': 0.5255388003741269, 'reg_lambda': 9.044891643391564}

Best CV R2:
0.8967237218308932


In [29]:
best_xgb = XGBRegressor(
    **study.best_params,

    objective='reg:squarederror',

    random_state=42
)


best_xgb.fit(
    X_train_scaled,
    y_train,
    verbose=False
)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.6988253984127397
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import

In [31]:
y_pred_train = best_xgb.predict(X_train_scaled)

y_pred_test = best_xgb.predict(X_test_scaled)


def evaluate_regression(y_true, y_pred):

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    r2 = r2_score(y_true, y_pred)

    mse = mean_squared_error(y_true, y_pred)

    rmse = np.sqrt(mse)

    mae = mean_absolute_error(y_true, y_pred)

    mape = np.mean(
        np.abs((y_true - y_pred) / (y_true + 1e-8))
    ) * 100

    evs = explained_variance_score(y_true, y_pred)

    pearson_r, _ = pearsonr(y_true, y_pred)

    relative_rmse = (rmse / np.mean(y_true)) * 100

    return {
        "R2": r2,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE (%)": mape,
        "Explained Variance Score": evs,
        "Pearson Correlation": pearson_r,
        "Relative RMSE (%)": relative_rmse
    }


train_metrics = evaluate_regression(y_train, y_pred_train)

test_metrics = evaluate_regression(y_test, y_pred_test)



df_results = pd.DataFrame({
    "Train": train_metrics,
    "Test": test_metrics
})


print(df_results.round(4))



                           Train    Test
R2                        0.9910  0.9568
MSE                       0.9650  6.2649
RMSE                      0.9824  2.5030
MAE                       0.6906  1.7587
MAPE (%)                  1.7040  4.7700
Explained Variance Score  0.9910  0.9580
Pearson Correlation       0.9955  0.9788
Relative RMSE (%)         2.1522  5.6623


In [32]:
#Save XGBoost model using pickle

import pickle

# save trained model
with open("D:\\APSirCNTandLSTM\\CNT with LSTM\\practice_Finalized For LSTM-CNT Scripts\\practice_LSTM and ANN_for publication\\LSTM Improved\\EnsemblesANNLSTM9010\\XGBoost\Saved models\\xgb_8020_updated.pkl", "wb") as f:
    pickle.dump(best_xgb, f)

print("XGBoost model saved successfully.")

XGBoost model saved successfully.


In [33]:
import pickle

# save trained model
with open("D:\\APSirCNTandLSTM\\CNT with LSTM\\practice_Finalized For LSTM-CNT Scripts\\practice_LSTM and ANN_for publication\\LSTM Improved\\EnsemblesANNLSTM9010\\XGBoost\\Saved models\\xgb_scalar8020updated.pkl", "wb") as f:
    pickle.dump(scaler_X_XGB, f)

print("XGBoost scalar saved successfully.")

XGBoost scalar saved successfully.


In [34]:
#Load model back

import pickle

with open("D:\\APSirCNTandLSTM\\CNT with LSTM\\practice_Finalized For LSTM-CNT Scripts\\practice_LSTM and ANN_for publication\\LSTM Improved\\EnsemblesANNLSTM9010\\XGBoost\Saved models\\xgb_8020_updated.pkl", "rb") as f:
    loaded_xgb = pickle.load(f)

print("Model loaded successfully.")

Model loaded successfully.


In [35]:
#Recompute predictions (train / val / test)

y_pred_train_loaded = loaded_xgb.predict(X_train_scaled)
y_pred_test_loaded  = loaded_xgb.predict(X_test_scaled)

In [37]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error,
    explained_variance_score
)

from scipy.stats import pearsonr


def evaluate(y_true, y_pred):

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    r2 = r2_score(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    evs = explained_variance_score(y_true, y_pred)
    pearson_r, _ = pearsonr(y_true, y_pred)
    rrmse = (rmse / np.mean(y_true)) * 100

    return {
        "R2": r2,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE (%)": mape,
        "Explained Variance": evs,
        "Pearson Correlation": pearson_r,
        "Relative RMSE (%)": rrmse
    }

train_metrics_loaded = evaluate(y_train, y_pred_train_loaded)
test_metrics_loaded  = evaluate(y_test, y_pred_test_loaded)


results_df = pd.DataFrame({
    "Train": train_metrics_loaded,
    "Test": test_metrics_loaded
})

print(results_df.round(4))

                      Train    Test
R2                   0.9910  0.9568
MSE                  0.9650  6.2649
RMSE                 0.9824  2.5030
MAE                  0.6906  1.7587
MAPE (%)             1.7040  4.7700
Explained Variance   0.9910  0.9580
Pearson Correlation  0.9955  0.9788
Relative RMSE (%)    2.1522  5.6623
